In [ ]:
from game import Connect4
import numpy as np
import math
import time

In [ ]:
connect4 = Connect4()

In [ ]:
state = connect4.get_initial_state()
player = 1 
while True:
    legal_moves = connect4.get_valid_moves(state)
    print(f"Legal Moves: {[i for i in range(connect4.col_count) if legal_moves[i]]}")
    
    action = int(input(f"player {player}: "))
    if action < 0 or action >= connect4.col_count or not legal_moves[action]:
        print('illegal move')
        continue

    state = connect4.make_move(state, player, action)
    connect4.print_board(state)

    value, terminated = connect4.get_value_and_terminated(state, action)

    if terminated:
        if value == 1:
            print(f"Player {player} wins!")
        else:
            print("draw")
        break


    player = connect4.get_opponent(player)


In [ ]:
def sample_move(valid_moves):
    chosen_move = np.random.choice(np.flatnonzero(valid_moves))
    #print(f"playing move {chosen_move}")
    return chosen_move

class Node:
    def __init__(self, args, game, state, player, action_taken=None, parent = None) -> None:

        self.game = game
        self.args = args
        self.state = state.copy()
        self.player = player

        self.visit_count = 0
        self.win_count = 0
        self.parent = parent
        self.children = []
        self.untried_actions = self.game.get_valid_moves(self.state) # this looks like: [True, True, False, ...]
        self.action_taken = action_taken


    def visit(self):
        self.visit_count += 1

    def is_terminal(self):
        if not self.action_taken:
            return False
        
        _, terminated = self.game.get_value_and_terminated(self.state, self.action_taken)
        return terminated

    def is_fully_expanded(self):
        return np.sum(self.untried_actions) == 0

    def has_children(self):
        return len(self.children) > 0

    def best_child(self):
        if np.sum(self.untried_actions) != 0:
            raise "need to explore all children first"
        
        if len(self.children) == 0:
            raise "no children to explore!"

        for child in self.children:
            if child.visit_count == 0:
                return child

        def ucb(child):
            exploit = child.win_count / child.visit_count
            explore = self.args['c'] * math.sqrt(math.log(self.visit_count) / child.visit_count)
            return exploit + explore

        return max(self.children, key=ucb)
    
    def expand(self):
        if self.is_fully_expanded():
            raise

        sampled_action = sample_move(self.untried_actions)
        #print(f"untried actions durign expansion: {self.untried_actions}")
        self.untried_actions[sampled_action] = False
        state = self.game.make_move(self.state, self.player, sampled_action)
        
        child_node = Node(
            args = self.args, 
            game = self.game,
            state = state,
            player = self.game.get_opponent(self.player),
            action_taken=sampled_action,
            parent=self
        )

        self.children.append(child_node)

        return child_node

    
    def rollout(self):
        
        state = self.state
        action_taken = self.action_taken
        player = self.player

        while True:
            value, terminated = self.game.get_value_and_terminated(state, action_taken)
            if terminated:
                #print(f"game finished, returning {value * player * -1}")
                return value * player * -1

            valid_moves = self.game.get_valid_moves(state)
            action_taken = sample_move(valid_moves)

            state = self.game.make_move(state, player, action_taken)
            player = self.game.get_opponent(player)

            #self.game.print_board(state)

        


    def UCB(self):
        if self.visit_count == 0:
            return np.inf
        pass

    def backpropogate(self, result):
        self.win_count += result
        self.visit()

        if self.parent is not None:
            self.parent.backpropogate(-1 * result)
        

In [ ]:
class MCTS:
    def __init__(self, game, args) -> None:
        self.game = game
        self.args = args

    def search(self, state, verbose = False):
       
        # define root
        root = Node(self.args, self.game, state, player=1)

        for i in range(self.args['iterations']):
            if verbose:
                #time.sleep(1)
                print(f"iteration {i} started")
            current_node = root
            
            while not current_node.is_terminal() and current_node.is_fully_expanded():
                current_node = current_node.best_child()
                if verbose:
                    self.game.print_board(current_node.state)
            
            if not current_node.is_terminal():
                current_node = current_node.expand()
            
            result = current_node.rollout()
            
            current_node.backpropogate(result)
            if verbose:
                print(f"iteration {i} finished, result = {result}")
        
        return root


In [ ]:
game = Connect4()
state_init = game.get_initial_state()

round_1 = MCTS(game=game, args={'iterations' : 400, 'c' : 2})
results = round_1.search(state=state_init, verbose=False)

In [ ]:
for child_node in results.children:
    print(f"action taken : {child_node.action_taken} \n num_visits: {child_node.visit_count} \n win count: {child_node.win_count}")

In [ ]:
import numpy as np

# Sample data
bool_arr = np.array([False, True, False, True, True, False, True])

# 1. Get the indices where the value is True
true_indices = np.flatnonzero(bool_arr)


# 3. Take a random sample of 2 indices without replacement
sampled_indices = np.random.choice(true_indices, size=2, replace=False)

print("Sampled Indices:", sampled_indices)


In [ ]:
print(state)

In [ ]:
row_count = 6
col_count = 7
wc = 4
import numpy as np

def check_win_test(state, action) -> bool:
    """True denotes a win, False means no winners"""
    
    padded_rows = row_count + 2 * (wc - 1)
    padded_cols = col_count + 2 * (wc - 1)
    
    padded_state = np.zeros([padded_rows, padded_cols])
    padded_state[wc - 1 : row_count + wc - 1, wc - 1 : col_count + wc - 1] = state

    col = action + wc -1
    row = None
    player = None

    for i in range(len(padded_state)):
        if padded_state[i, col] != 0:
            player = padded_state[i, col]
            row = i
            break

    print(col)
    print(row)
    print(player)
    print(padded_state)

    print(padded_state[row, col : col - wc])


    if np.sum(padded_state[row - wc : row, col]) == player * wc: #up: 
        print("up")
    elif np.sum(padded_state[row : row + wc, col]) == player * wc: # down
        print("down")
    elif np.sum([padded_state[row + i, col + i] for i in range(wc)]) == player * wc: # down right
        print("d r")
    elif np.sum([padded_state[row + i, col - i] for i in range(wc)]) == player * wc: # down left
        print("dl")
    elif np.sum([padded_state[row - i, col + i] for i in range(wc)]) == player * wc: # up right
        print('u r')
    elif np.sum([padded_state[row - i, col - i] for i in range(wc)]) == player * wc: # up left
        print("u l")

    if np.sum(padded_state[row, col - wc + 1 : col + 1]) == player * wc or np.sum(padded_state[row, col : col + wc]) == player * wc:
        print("works")

In [ ]:
check_win_test(state, 5)

In [ ]:
state